## 4. NASA HLS Imagery → NDVI & EVI

We use `earthaccess` to search NASA's cloud archive for HLS scenes over each state at each of the four forecast date windows. For each scene we compute NDVI and EVI, apply cloud masking (Fmask), and save state-level mean values.

**You need a NASA Earthdata account** — register free at https://urs.earthdata.nasa.gov/

In [2]:
# Authenticate with NASA Earthdata
# First time: will prompt for username/password and save to ~/.netrc
auth = earthaccess.login(persist=True)
print('✓ Earthdata authenticated')

NameError: name 'earthaccess' is not defined

In [13]:
# Configure GDAL for cloud-native access (from the HLS tutorial)
cookie_file_path = os.path.expanduser('~/cookies.txt')
gdal.SetConfigOption('GDAL_HTTP_COOKIEFILE', cookie_file_path)
gdal.SetConfigOption('GDAL_HTTP_COOKIEJAR',  cookie_file_path)
gdal.SetConfigOption('GDAL_DISABLE_READDIR_ON_OPEN', 'EMPTY_DIR')
gdal.SetConfigOption('CPL_VSIL_CURL_ALLOWED_EXTENSIONS', 'TIF')
gdal.SetConfigOption('GDAL_HTTP_UNSAFESSL', 'YES')
gdal.SetConfigOption('GDAL_HTTP_MAX_RETRY', '10')
gdal.SetConfigOption('GDAL_HTTP_RETRY_DELAY', '0.5')

os.environ['GDAL_HTTP_COOKIEFILE'] = cookie_file_path
os.environ['GDAL_HTTP_COOKIEJAR']  = cookie_file_path

print('✓ GDAL configured for cloud access')

✓ GDAL configured for cloud access


In [14]:
# ── Helper functions (adapted from HLS_Tutorial.ipynb) ───────────────────────

def get_band_links(granule_urls, product_type):
    """Filter HLS URLs to just the bands needed for EVI + NDVI."""
    if 'HLSS30' in product_type:
        bands = ['B8A', 'B04', 'B02', 'B03', 'Fmask']  # NIR, Red, Blue, Green
    else:
        bands = ['B05', 'B04', 'B02', 'B03', 'Fmask']  # Landsat
    return [u for u in granule_urls if any(b in u for b in bands)]


def create_quality_mask(quality_data, bit_nums=[1, 2, 3, 4, 5]):
    """Build a bad-pixel mask from the HLS Fmask layer."""
    mask_array = np.zeros(quality_data.shape[-2:], dtype=bool)
    q = np.nan_to_num(np.array(quality_data), 255).astype(np.int8)
    if q.ndim == 3:
        q = q[0]
    for bit in bit_nums:
        mask_array = np.logical_or(mask_array, (q & (1 << bit)) > 0)
    return mask_array


def scaling(band):
    """Apply HLS scale factor of 0.0001."""
    band_out = band.copy()
    band_out.data = band.data * 0.0001
    return band_out


def calc_ndvi(nir, red):
    """NDVI = (NIR - Red) / (NIR + Red)"""
    ndvi = (nir - red) / (nir + red + 1e-8)
    return ndvi.clip(-1, 1)


def calc_evi(nir, red, blue):
    """EVI = 2.5 * (NIR - Red) / (NIR + 6*Red - 7.5*Blue + 1)"""
    evi = 2.5 * ((nir - red) / (nir + 6.0 * red - 7.5 * blue + 1.0 + 1e-8))
    evi = xr.where(evi != np.inf, evi, np.nan, keep_attrs=True)
    return evi.clip(-1, 1)


def load_and_compute_vi(granule_urls, bbox_4326, product_type):
    """
    Loads one HLS granule, computes NDVI and EVI masked to the state bbox.
    Returns dict with mean NDVI and EVI (floats), or None on failure.
    """
    chunk = dict(band=1, x=512, y=512)
    bands = {}
    
    if 'HLSS30' in product_type:
        band_map = {'nir': 'B8A', 'red': 'B04', 'blue': 'B02', 'fmask': 'Fmask'}
    else:
        band_map = {'nir': 'B05', 'red': 'B04', 'blue': 'B02', 'fmask': 'Fmask'}
    
    try:
        for role, band_code in band_map.items():
            url = next((u for u in granule_urls if f'.{band_code}.' in u), None)
            if url is None:
                return None
            da = rxr.open_rasterio(url, chunks=chunk, masked=True).squeeze('band', drop=True)
            # Reproject to WGS84 and clip to state bbox
            da = da.rio.reproject('EPSG:4326')
            lon_min, lat_min, lon_max, lat_max = bbox_4326
            da = da.rio.clip_box(minx=lon_min, miny=lat_min, maxx=lon_max, maxy=lat_max)
            bands[role] = da
        
        # Scale reflectance bands
        nir  = scaling(bands['nir'])
        red  = scaling(bands['red'])
        blue = scaling(bands['blue'])
        
        # Quality mask
        qmask = create_quality_mask(bands['fmask'].values)
        
        # Compute indices
        ndvi = calc_ndvi(nir, red)
        evi  = calc_evi(nir, red, blue)
        
        # Apply mask
        ndvi = ndvi.where(~qmask)
        evi  = evi.where(~qmask)
        
        # Return mean values (compute triggers dask)
        return {
            'ndvi_mean': float(ndvi.mean().compute()),
            'ndvi_std':  float(ndvi.std().compute()),
            'evi_mean':  float(evi.mean().compute()),
            'evi_std':   float(evi.std().compute()),
            'valid_pixels': int((~np.isnan(ndvi.values)).sum()),
        }
    except Exception as e:
        print(f'    ⚠️  Error: {e}')
        return None

print('✓ Helper functions defined')

✓ Helper functions defined


In [16]:
# ── Fetch 2025 forecast data ──────────────────────────────────────────────────
print('=== Fetching 2025 HLS data (forecast year) ===')
print(f'States: {list(STATE_BBOX.keys())}')
print(f'Date windows: {list(FORECAST_WINDOWS.keys())}')
print()

for state, bbox in STATE_BBOX.items():
    for date_label, (t_start, t_end) in FORECAST_WINDOWS.items():
        vi_stats = fetch_hls_vi_for_window(state, bbox, (t_start, t_end), 2025, date_label)
        row = {
            'state': state,
            'year': 2025,
            'forecast_date': date_label,
            'is_forecast': True,
        }
        if vi_stats:
            row.update(vi_stats)
        records.append(row)

print()
print(f'✓ 2025 HLS fetch complete — {len(records)} records so far')

=== Fetching 2025 HLS data (forecast year) ===
States: ['Iowa', 'Colorado', 'Wisconsin', 'Missouri', 'Nebraska']
Date windows: ['aug1', 'sep1', 'oct1', 'final']

  Iowa aug1 2025: 5 granules found
  Iowa sep1 2025: 5 granules found
  Iowa oct1 2025: 5 granules found
  Iowa final 2025: 5 granules found
  Colorado aug1 2025: 5 granules found
  Colorado sep1 2025: 5 granules found
  Colorado oct1 2025: 5 granules found
  Colorado final 2025: 5 granules found
  Wisconsin aug1 2025: 5 granules found
  Wisconsin sep1 2025: 5 granules found
  Wisconsin oct1 2025: 5 granules found
  Wisconsin final 2025: 5 granules found
  Missouri aug1 2025: 5 granules found
  Missouri sep1 2025: 5 granules found
  Missouri oct1 2025: 5 granules found
  Missouri final 2025: 5 granules found
  Nebraska aug1 2025: 5 granules found
  Nebraska sep1 2025: 5 granules found
  Nebraska oct1 2025: 5 granules found
  Nebraska final 2025: 5 granules found

✓ 2025 HLS fetch complete — 20 records so far


In [18]:
import os, pandas as pd
os.makedirs('data/hls', exist_ok=True)
df_2025 = pd.DataFrame(records)
df_2025.to_csv('data/hls/hls_2025.csv', index=False)
print(f'Saved {len(df_2025)} records')

Saved 23 records


In [3]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray as rxr
import matplotlib.pyplot as plt
import earthaccess
from osgeo import gdal
from shapely.geometry import box

warnings.filterwarnings('ignore')

STATE_BBOX = {
    'Iowa':      (-96.7, 40.3, -90.1, 43.6),
    'Colorado':  (-109.1, 36.9, -102.0, 41.1),
    'Wisconsin': (-92.9, 42.5, -86.8, 47.1),
    'Missouri':  (-95.8, 35.9, -89.1, 40.6),
    'Nebraska':  (-104.1, 39.9, -95.3, 43.0),
}

cookie_file = os.path.expanduser('~/cookies.txt')
gdal.SetConfigOption('GDAL_HTTP_COOKIEFILE', cookie_file)
gdal.SetConfigOption('GDAL_HTTP_COOKIEJAR',  cookie_file)
gdal.SetConfigOption('GDAL_DISABLE_READDIR_ON_OPEN', 'EMPTY_DIR')
gdal.SetConfigOption('CPL_VSIL_CURL_ALLOWED_EXTENSIONS', 'TIF')
gdal.SetConfigOption('GDAL_HTTP_UNSAFESSL', 'YES')
gdal.SetConfigOption('GDAL_HTTP_MAX_RETRY', '10')
gdal.SetConfigOption('GDAL_HTTP_RETRY_DELAY', '0.5')
os.environ['GDAL_HTTP_COOKIEFILE'] = cookie_file
os.environ['GDAL_HTTP_COOKIEJAR']  = cookie_file

def create_quality_mask(quality_data, bit_nums=[1,2,3,4,5]):
    mask = np.zeros(quality_data.shape[-2:], dtype=bool)
    q = np.nan_to_num(np.array(quality_data), 255).astype(np.int8)
    if q.ndim == 3: q = q[0]
    for bit in bit_nums:
        mask = np.logical_or(mask, (q & (1 << bit)) > 0)
    return mask

def scaling(band):
    out = band.copy()
    out.data = band.data * 0.0001
    return out

def calc_ndvi(nir, red):
    return ((nir - red) / (nir + red + 1e-8)).clip(-1, 1)

def calc_evi(nir, red, blue):
    evi = 2.5 * ((nir - red) / (nir + 6.0*red - 7.5*blue + 1.0 + 1e-8))
    return evi.clip(-1, 1)

def load_and_compute_vi(urls, bbox, product_type):
    chunk = dict(band=1, x=512, y=512)
    band_map = {'nir': 'B8A', 'red': 'B04', 'blue': 'B02', 'fmask': 'Fmask'} \
               if 'HLSS30' in product_type else \
               {'nir': 'B05', 'red': 'B04', 'blue': 'B02', 'fmask': 'Fmask'}
    try:
        bands = {}
        lon_min, lat_min, lon_max, lat_max = bbox
        for role, code in band_map.items():
            url = next((u for u in urls if f'.{code}.' in u), None)
            if not url: return None
            da = rxr.open_rasterio(url, chunks=chunk, masked=True).squeeze('band', drop=True)
            da = da.rio.reproject('EPSG:4326')
            da = da.rio.clip_box(minx=lon_min, miny=lat_min, maxx=lon_max, maxy=lat_max)
            bands[role] = da
        nir  = scaling(bands['nir'])
        red  = scaling(bands['red'])
        blue = scaling(bands['blue'])
        qmask = create_quality_mask(bands['fmask'].values)
        ndvi = calc_ndvi(nir, red).where(~qmask)
        evi  = calc_evi(nir, red, blue).where(~qmask)
        return {
            'ndvi_mean': float(ndvi.mean().compute()),
            'ndvi_std':  float(ndvi.std().compute()),
            'evi_mean':  float(evi.mean().compute()),
            'evi_std':   float(evi.std().compute()),
            'n_granules': 1,
        }
    except Exception as e:
        print(f'    Error: {e}')
        return None

def fetch_hls_vi_for_window(state, bbox, temporal_window, year, label):
    results = earthaccess.search_data(
        short_name=['HLSL30', 'HLSS30'],
        bounding_box=bbox,
        temporal=temporal_window,
        count=5
    )
    if not results:
        print(f'    No granules: {state} {label} {year}')
        return None
    print(f'  {state} {label} {year}: {len(results)} granules')
    vi_list = []
    for g in results:
        urls = g.data_links()
        product = 'HLSS30' if any('HLSS30' in u for u in urls) else 'HLSL30'
        vi = load_and_compute_vi(urls, bbox, product)
        if vi: vi_list.append(vi)
    if not vi_list: return None
    return {
        'ndvi_mean': float(np.median([v['ndvi_mean'] for v in vi_list])),
        'ndvi_std':  float(np.median([v['ndvi_std']  for v in vi_list])),
        'evi_mean':  float(np.median([v['evi_mean']  for v in vi_list])),
        'evi_std':   float(np.median([v['evi_std']   for v in vi_list])),
        'n_granules': len(vi_list),
    }

# Load existing 2025 data
records = pd.read_csv('data/hls/hls_2025.csv').to_dict('records')
print(f'✓ Setup complete — loaded {len(records)} existing 2025 records')

earthaccess.login(persist=True)

✓ Setup complete — loaded 23 existing 2025 records


In [ ]:
# ── Fetch historical HLS data (training years 2013–2024) ─────────────────────
# Note: this loop can take 30–60 minutes for all years × states × dates.
# If time is short, limit HISTORICAL_YEARS to fewer years (e.g. 2018–2024).

HISTORICAL_YEARS_TO_FETCH = list(range(2018, 2025))  # reduce for speed

print(f'=== Fetching historical HLS data ({HISTORICAL_YEARS_TO_FETCH[0]}–{HISTORICAL_YEARS_TO_FETCH[-1]}) ===')
print('Note: this will take a while. Progress is saved to CSV after each year.')
print()

hist_windows = {
    'aug1':  ('07-17', '08-15'),
    'sep1':  ('08-17', '09-15'),
    'oct1':  ('09-17', '10-15'),
    'final': ('10-17', '11-15'),
}

for year in HISTORICAL_YEARS_TO_FETCH:
    print(f'--- Year {year} ---')
    for state, bbox in STATE_BBOX.items():
        for date_label, (m_start, m_end) in hist_windows.items():
            t_start = f'{year}-{m_start}'
            t_end   = f'{year}-{m_end}'
            vi_stats = fetch_hls_vi_for_window(state, bbox, (t_start, t_end), year, date_label)
            row = {
                'state': state,
                'year': year,
                'forecast_date': date_label,
                'is_forecast': False,
            }
            if vi_stats:
                row.update(vi_stats)
            records.append(row)
    
    # Save checkpoint after each year
    df_vi = pd.DataFrame(records)
    df_vi.to_csv('data/hls/hls_vi_features.csv', index=False)
    print(f'  → Checkpoint saved ({len(df_vi)} records)')

print()
print(f'✓ HLS fetch complete — {len(records)} total records')

=== Fetching historical HLS data (2018–2024) ===
Note: this will take a while. Progress is saved to CSV after each year.

--- Year 2018 ---
  Iowa aug1 2018: 5 granules
  Iowa sep1 2018: 5 granules
  Iowa oct1 2018: 5 granules
  Iowa final 2018: 5 granules
  Colorado aug1 2018: 5 granules
  Colorado sep1 2018: 5 granules
  Colorado oct1 2018: 5 granules
  Colorado final 2018: 5 granules
  Wisconsin aug1 2018: 5 granules
  Wisconsin sep1 2018: 5 granules
  Wisconsin oct1 2018: 5 granules
  Wisconsin final 2018: 5 granules
  Missouri aug1 2018: 5 granules
  Missouri sep1 2018: 5 granules
  Missouri oct1 2018: 5 granules
  Missouri final 2018: 5 granules
  Nebraska aug1 2018: 5 granules
  Nebraska sep1 2018: 5 granules
  Nebraska oct1 2018: 5 granules
  Nebraska final 2018: 5 granules
  → Checkpoint saved (43 records)
--- Year 2019 ---
  Iowa aug1 2019: 5 granules
  Iowa sep1 2019: 5 granules
  Iowa oct1 2019: 5 granules
  Iowa final 2019: 5 granules
  Colorado aug1 2019: 5 granules
  Col

In [ ]:
# Save final HLS VI feature table
df_vi = pd.DataFrame(records)
df_vi.to_csv('data/hls/hls_vi_features.csv', index=False)
print('✓ Saved: data/hls/hls_vi_features.csv')
print(df_vi.head(10))